# ConQRR (2020)
[[paper]](https://arxiv.org/abs/2105.08811)<br>
ConQRR = Conversational Query Rewriting for Retrieval

**ConQRR** — это метод переписывания поисковых запросов в контексте многораундовых диалогов. Модель обучается преобразовывать неполный или двусмысленный реплики пользователя (например, содержащие местоимения «он», «этот») в полноценные, семантически независимые запросы, оптимизированные специально для поисковых систем.

**Контекст**<br>
В диалоговых системах (Information-seeking dialogue) пользователи часто полагаются на контекст предыдущих фраз. Возникают две основные проблемы:
1.  Anaphora: использование местоимений, ссылающихся на объекты из истории («Расскажи о его биографии»).
2.  Ellipsis: пропуск слов, которые подразумеваются из контекста («А где он родился?» — подразумевается «Где родился [ученый X]?»).

**Идея**<br>
Большинство существующих методов обучаются на основе Maximum Likelihood Estimation (MLE), чтобы минимизировать разницу между предсказанием модели и эталонным текстом, написанным человеком. Однако «человеческое» переписывание не всегда является оптимальным для поискового движка. Идея ConQRR — использовать Reinforcement Learning (RL), где награда (reward) напрямую зависит от качества работы поисковика (Retriever) после переписывания.

**Постановка задачи**<br>
Дан текущий запрос $q_t$ и история диалога $H = \{q_1, a_1, ..., q_{t-1}, a_{t-1}\}$. Нужно сгенерировать новый запрос $q'_t$, который содержит всю необходимую информацию для поиска релевантного документа $D^+$ в большой коллекции без обращения к истории $H$.

**Альтернативы на 2020 год**<br>
- **Query Likelihood / Expansion**: классические методы добавления терминов из истории, часто приводили к зашумлению запроса лишними словами.
- **Original Query + Context**: простая конкатенация истории и текущего запроса для энкодера. Неэффективно, так как увеличивает длину входа и вносит много нерелевантного шума.
- **GPT-2 / T5 Rewriters (SFT)**: модели, обученные только с учителем (Supervised Fine-Tuning) на датасетах типа CANARD. Проблема: модель может выдать грамматически верный текст, который плохо работает с BM25 или DPR из-за отсутствия ключевых ключевых слов.

**Архитектура**<br>
Система состоит из двух основных блоков:
1.  **Rewriter**: генеративная модель (в оригинале — T5-base), которая принимает на вход $H + q_t$ и генерирует последовательность токенов $q'_t$.
2.  **Scorer (Environment)**: фиксированный Retriever (например, BM25 или обученный DPR (2020)), который оценивает, насколько удачным получился переписанный запрос.

**Алгоритм обучения**<br>
Процесс разбит на две стадии:

1.  **Warm-up (SFT)**:
    Модель обучается на парах «контекст + запрос —> эталонный переписанный запрос» из датасета CANARD. Это необходимо, чтобы генератор научился основам языка и не выдавал бессмысленный набор токенов на старте RL.

2.  **Reinforcement Learning (Policy Gradient)**:
    - Модель генерирует несколько вариантов переписанного запроса с помощью Sampling.
    - Для каждого варианта вычисляется награда на основе метрик ранжирования.
    - Используется алгоритм REINFORCE. Для уменьшения дисперсии градиента применяется Baseline (средняя награда или награда от жадного декодирования).
    - **Reward Shaping**: Награда $R$ считается как Recall@K или MRR. Для того чтобы модель не "ломала" язык в угоду поисковику, добавляется штраф в виде KL-divergence между текущей политикой и SFT-моделью.

**Алгоритм инференса**<br>
1. На вход подается цепочка диалога и текущая реплика.
2. Rewriter генерирует переписанный запрос $q'_t$ (используется Beam Search для стабильности).
3. Полученный $q'_t$ отправляется в стандартный Dense или Sparse Retriever.
4. Извлеченные документы выдаются пользователю или передаются в Reader-модель.

**Результаты**<br>
Эксперименты проводились на датасетах TREC CAsT (2019) и QReCC.
- **Эффективность**: Использование RL-этапа позволило улучшить метрику Recall@100 на 6-8 п.п. по сравнению с чистым SFT-обучением на той же архитектуре (T5).
- **Влияние на Retrieval**: ConQRR показал, что запросы, оптимизированные через RL, часто содержат избыточные синонимы или специфические ключевые слова, которые человек бы не вставил в "красивое" предложение, но которые критически важны для работы BM25.
- **Сравнение**: По метрике MRR модель превзошла существовавший на тот момент метод QuReTeC (2020) на 12%, за счет того, что QuReTeC был ограничен только операциями классификации токенов (оставлять/удалять), а ConQRR может генерировать новые связующие слова.

## 📝 Критический анализ

```markdown
# ConQRR (2020)
---
[[paper]](https://arxiv.org/abs/2105.08811)<br>
ConQRR = Conversational Query Rewriting for Retrieval

**ConQRR** — метод переписывания запросов в диалогах. Модель преобразует неполные реплики пользователя в полноценные запросы для поисковых систем.

**Контекст**<br>
В диалогах пользователи часто используют анафоры и эллипсисы, что затрудняет поиск. ConQRR решает эти проблемы.

**Идея**<br>
В отличие от методов на основе MLE, ConQRR использует Reinforcement Learning (RL), где награда зависит от качества работы поисковика после переписывания.

**Постановка задачи**<br>
Дан запрос $q_t$ и история $H$. Нужно сгенерировать $q'_t$, содержащий всю информацию для поиска документа $D^+$ без обращения к $H$.

**Альтернативы на 2020 год**<br>
- **Query Likelihood / Expansion**: добавление терминов из истории, часто зашумляющее запрос.
- **Original Query + Context**: конкатенация истории и запроса, увеличивающая длину и шум.
- **GPT-2 / T5 Rewriters (SFT)**: модели, обученные с учителем, но не всегда эффективные для BM25 или DPR.

**Архитектура**<br>
Система состоит из:
1. **Rewriter**: генеративная модель (T5-base), генерирующая $q'_t$.
2. **Scorer (Environment)**: Retriever (например, BM25 или DPR), оценивающий переписанный запрос.

<img src="img/img.png" width=500>

**Алгоритм обучения**<br>
1. **Warm-up (SFT)**: обучение на парах «контекст + запрос —> эталонный запрос» из CANARD.
2. **Reinforcement Learning (Policy Gradient)**:
   - Генерация вариантов переписанного запроса.
   - Вычисление награды на основе метрик ранжирования.
   - Алгоритм REINFORCE с Baseline для уменьшения дисперсии.
   - **Reward Shaping**: награда $R$ как Recall@K или MRR с штрафом за отклонение от SFT.

**Алгоритм инференса**<br>
1. Вход: диалог и текущая реплика.
2. Rewriter генерирует $q'_t$ (Beam Search).
3. $q'_t$ отправляется в Retriever.
4. Извлеченные документы выдаются пользователю или Reader-модели.

**Результаты**<br>
- **Эффективность**: RL-этап улучшил Recall@100 на 6-8 п.п. по сравнению с SFT.
- **Влияние на Retrieval**: запросы через RL содержат важные ключевые слова для BM25.
- **Сравнение**: ConQRR превзошел QuReTeC (2020) на 12% по MRR, генерируя новые связующие слова.
```

## 💻 Пример кода

Иллюстративный Python пример, демонстрирующий основные концепции:

In [ ]:
# Пример реализации основных концепций ConQRR на Python

# Импортируем необходимые библиотеки
import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
from rank_bm25 import BM25Okapi

# Инициализация модели и токенизатора T5
tokenizer = T5Tokenizer.from_pretrained("t5-base")
model = T5ForConditionalGeneration.from_pretrained("t5-base")

# Пример истории диалога и текущего запроса
history = [
    "Who is Albert Einstein?",
    "Tell me about his biography.",
    "Where was he born?"
]
current_query = "And what about his achievements?"

# Конкатенация истории и текущего запроса
input_text = " ".join(history) + " " + current_query

# Токенизация входного текста
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# Генерация переписанного запроса с помощью T5
output_ids = model.generate(input_ids, num_beams=5, max_length=50)
rewritten_query = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Rewritten Query:", rewritten_query)

# Пример использования BM25 для оценки качества переписанного запроса
# Создаем коллекцию документов (игрушечный пример)
documents = [
    "Albert Einstein was born in Ulm, Germany.",
    "He developed the theory of relativity.",
    "Einstein received the Nobel Prize in Physics in 1921."
]

# Токенизация документов
tokenized_docs = [doc.split(" ") for doc in documents]

# Инициализация BM25
bm25 = BM25Okapi(tokenized_docs)

# Оценка переписанного запроса
tokenized_query = rewritten_query.split(" ")
scores = bm25.get_scores(tokenized_query)

print("BM25 Scores:", scores)

# Пример использования Reinforcement Learning для улучшения переписанного запроса
# (упрощенная версия, без реализации полного RL цикла)

# Функция награды, основанная на BM25
def reward_function(query, documents):
    tokenized_query = query.split(" ")
    scores = bm25.get_scores(tokenized_query)
    return sum(scores)  # Простая сумма баллов BM25 как награда

# Генерация нескольких вариантов переписанного запроса
sampled_queries = [
    tokenizer.decode(model.generate(input_ids, num_beams=5, max_length=50, do_sample=True)[0], skip_special_tokens=True)
    for _ in range(5)
]

# Вычисление награды для каждого варианта
rewards = [reward_function(q, documents) for q in sampled_queries]

# Выбор лучшего варианта на основе награды
best_query = sampled_queries[rewards.index(max(rewards))]

print("Best Rewritten Query:", best_query)

# Важно отметить, что в реальной реализации RL этапа используется алгоритм REINFORCE
# и более сложные техники, такие как Reward Shaping и Baseline для уменьшения дисперсии градиента.
```

### Комментарии к коду:
1. **Rewriter**: Используется модель T5 для переписывания запроса. Мы конкатенируем историю диалога и текущий запрос, чтобы модель могла учитывать контекст.
2. **Scorer (Environment)**: Используем BM25 для оценки качества переписанного запроса. Это иллюстрирует, как ConQRR использует фиксированный Retriever для оценки.
3. **Reinforcement Learning**: Пример упрощенной функции награды, основанной на BM25. В реальной реализации используется алгоритм REINFORCE и другие техники для оптимизации переписывания.
4. **Inference**: Генерация и выбор лучшего переписанного запроса на основе награды.